# Computational Theory: SHA-256

## Problem 1: Representing SHA-256 Data

### 32-bit words
Before beginning with any proper SHA-256 implementation I'll start with looking into how the input, output and intermediate data can be represented in Python. 

In [ ]:
# 32-bit words.

# largest 32-bit value.
mask = 0xFFFFFFFF

# The largest unsigned 32-bit value, and 1 to add to it.
hello = 0xFFFFFFFF
bye = 1

# The sum is 0x100000000, which needs 33 bits. ANDing it with the mask
# drops bit 32, so the result wraps to 0.
result = (hello + bye) & mask

# Printing value as hex.
print(f"0x{result:08X}")

0x00000000


### Sequences of 32-bit words

SHA-256 works with 32 bit words the whole way through. Everything it uses comes in fixed size groups of them: the 16 words of a message block, the 64-word message schedule, the 64 round constants, and the 8 words of the hash state.

I'll represent this using:

- a Python list of ints: every element is an unlimited size
  integer, so each one has to be masked manually to stay within 32 bits;

In [6]:
# Example of eight random 32 bit words.
words = [
    0x1a2b3c4d, 0xdeadbeef, 0x00000001, 0xffffffff,
    0x80000000, 0x12345678, 0x26d8aabe, 0x0badf00d,
]

# Add each word to itself, masking so we keep 32 bits.
doubled = [(w + w) & mask for w in h0]

# Print each value as hex
print([f"{w:08x}" for w in doubled])

['d413ccce', '76cf5d0a', '78dde6e4', '4a9fea74', 'a21ca4fe', '360ad118', '3f07b356', 'b7c19a32']


You can use a Python list of ints to represent sequences of 32 bit words, but since Python ints can grow large, you have to apply a mask after every operation to keep the values within 32 bits.

### Input Messages

SHA-256 takes a message as a sequence of bytes, so I can use Python bytes objects to represent input messages. Text has to be encoded first. I can use UTF-8, which turns ASCII characters into one byte each and other characters into several bytes.

In [7]:
# Encode text messages as UTF-8 bytes.
ascii_msg = "hello".encode("utf-8")
non_ascii_msg = "goodbye".encode("utf-8")

# Show each messages bytes in hex and its length in bits.
for msg in [ascii_msg, non_ascii_msg]:
    print(msg, msg.hex(), f"{len(msg) * 8} bits")

b'hello' 68656c6c6f 40 bits
b'goodbye' 676f6f64627965 56 bits


bytes is the right fit because SHA-256 hashes bytes and not characters. It's
immutable, its length in bits is just `len(msg) * 8`, and it's what `hashlib.sha256` accepts, so I can test against it.

### 512-bit message blocks

SHA-256 processes a message in 512 bit blocks. That's 64 bytes, or 16
32-bit words. I can represent a block as a list of 16 ints, built by splitting
64 bytes into 4 byte chunks and converting each chunk to a word.

In [ ]:
# A 64 byte block
block = bytes(range(64))

# Split into 4 byte chunks and then we convert each one to a 32 bit word.
block_words = [int.from_bytes(block[i:i + 4], "big") for i in range(0, 64, 4)]

print(f"{len(block) * 8} bits, {len(block_words)} words")
print([f"{w:08x}" for w in block_words])

A list of 16 words matches how the standard describes a block. Real messages are almost never a multiple of 64 bytes.

### The final 256-bit hash value

SHA-256 ends with eight 32 bit words of hash state. Put
together, they make up the 256-bit digest. I'll keep it as a list of 8 words while
computing, then can return it as a 64 character lowercase hex string.

In [8]:
# Final hash state for the message "abc".
final_words = [
    0xba7816bf, 0x8f01cfea, 0x414140de, 0x5dae2223,
    0xb00361a3, 0x96177a9c, 0xb410ff61, 0xf20015ad,
]

# Join the words as 8 digit hex strings so we can get the 64 character digest.
digest = "".join(f"{w:08x}" for w in final_words)

print(digest, f"({len(digest)} hex characters)")

# Expected SHA-256 digest of 'abc' from the standards examples.
assert digest == "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"

ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad (64 hex characters)


Eight words are handy while computing, since each one gets updated separately.
A hex string is also the standard way to show a digest since each hex digit is 4 bits, so
256 bits is always 64 characters.